# Metrics Evaluation: Baseline vs Multi-Agent

**Purpose**: Compare kg-axel baseline vs Multi-Agent system performance

**Inputs**:
- `output/baseline_kg_axel.csv` (from notebook 01)
- `output/multiagent_k3.csv` (from notebook 02)
- `output/multiagent_k3_states.json` (from notebook 02)

**Output**: `outputMetrics/comparison_metrics.csv` + visualizations

## Metrics Computed:

### Primary Metrics:
1. **Pass@k**: % queries where output matches ground truth
2. **KG Valid@k**: % queries that are valid Cypher and execute successfully
3. **Execution Success Rate**: % queries that execute without errors
4. **Empty Result Rate**: % queries that return empty results

### Multi-Agent Specific:
5. **Recovery Rate**: % queries that failed initially but succeeded after refinement
6. **Avg Iterations**: Average number of refinement iterations
7. **First-Attempt Success**: % queries that succeeded on first attempt

### Cost Metrics:
8. **Total Tokens**: Token consumption
9. **Avg Tokens per Question**: Token efficiency
10. **Latency**: Processing time

## Stratified Analysis:
- By Complexity (Easy/Medium/Hard)
- By Reasoning Level (Fakta Eksplisit/Implisit)
- By Sublevel (Nodes/One-hop/Multi-hop)

## 1. Setup

In [ ]:
import sys
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Setup plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print(f"Metrics Evaluation started: {datetime.now()}")
print(f"Working directory: {Path.cwd()}")

## 2. Load Results

In [ ]:
# Paths
OUTPUT_DIR = Path.cwd().parent / "output"
METRICS_DIR = Path.cwd().parent / "outputMetrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Load baseline results
baseline_path = OUTPUT_DIR / "baseline_kg_axel.csv"
baseline_df = pd.read_csv(baseline_path)
print(f"Loaded baseline: {len(baseline_df)} questions")

# Load multi-agent results
multiagent_path = OUTPUT_DIR / "multiagent_k3.csv"
multiagent_df = pd.read_csv(multiagent_path)
print(f"Loaded multi-agent: {len(multiagent_df)} questions")

# Load multi-agent states (for detailed analysis)
states_path = OUTPUT_DIR / "multiagent_k3_states.json"
with open(states_path, "r", encoding="utf-8") as f:
    ma_states = json.load(f)
print(f"Loaded {len(ma_states)} multi-agent execution states")

# Verify data alignment
assert len(baseline_df) == len(multiagent_df), "Baseline and multi-agent datasets must have same size"
print("\nData loaded successfully!")

## 3. Compute Pass@k Metric

Pass@k requires comparing execution outputs (not just query text)

In [ ]:
from evaluation.metrics import compute_pass_at_k

def compare_outputs(gt_result, generated_result):
    """
    Compare ground truth execution result with generated query result.
    
    Returns True if outputs match (same records, order-agnostic).
    """
    if not gt_result or not generated_result:
        return gt_result == generated_result
    
    # Parse JSON strings if needed
    if isinstance(gt_result, str):
        gt_result = json.loads(gt_result)
    if isinstance(generated_result, str):
        generated_result = json.loads(generated_result)
    
    # Convert to sorted tuples for comparison
    def normalize(result):
        if isinstance(result, list):
            return sorted([tuple(sorted(r.items())) if isinstance(r, dict) else r for r in result])
        return result
    
    return normalize(gt_result) == normalize(generated_result)

# Compute Pass@k for baseline
baseline_df['pass_at_1'] = baseline_df.apply(
    lambda row: compare_outputs(row.get('ground_truth_result'), row.get('execution_result')),
    axis=1
)

# Compute Pass@k for multi-agent
multiagent_df['pass_at_k'] = multiagent_df.apply(
    lambda row: compare_outputs(row.get('ground_truth_result'), row.get('execution_result')),
    axis=1
)

print("Pass@k computed successfully")
print(f"Baseline Pass@1: {baseline_df['pass_at_1'].mean():.2%}")
print(f"Multi-Agent Pass@k: {multiagent_df['pass_at_k'].mean():.2%}")

## 4. Primary Metrics Comparison

In [ ]:
# Calculate primary metrics
metrics = {
    'Metric': [],
    'Baseline (kg-axel)': [],
    'Multi-Agent': [],
    'Improvement': [],
    'Relative Improvement': []
}

def add_metric(name, baseline_val, multiagent_val, is_percentage=True):
    metrics['Metric'].append(name)
    metrics['Baseline (kg-axel)'].append(baseline_val)
    metrics['Multi-Agent'].append(multiagent_val)
    
    improvement = multiagent_val - baseline_val
    metrics['Improvement'].append(improvement)
    
    if baseline_val > 0:
        rel_improvement = (improvement / baseline_val) * 100
        metrics['Relative Improvement'].append(f"{rel_improvement:+.1f}%")
    else:
        metrics['Relative Improvement'].append("N/A")

# Primary metrics
add_metric(
    "Pass@k (%)",
    baseline_df['pass_at_1'].mean() * 100,
    multiagent_df['pass_at_k'].mean() * 100
)

add_metric(
    "KG Valid@k (%)",
    baseline_df['execution_success'].mean() * 100,
    multiagent_df['execution_success'].mean() * 100
)

add_metric(
    "Execution Success (%)",
    baseline_df['execution_success'].mean() * 100,
    multiagent_df['execution_success'].mean() * 100
)

add_metric(
    "Empty Result Rate (%)",
    baseline_df['is_empty_result'].mean() * 100,
    multiagent_df['is_empty_result'].mean() * 100
)

# Create DataFrame
metrics_df = pd.DataFrame(metrics)
print("\n" + "="*80)
print("PRIMARY METRICS COMPARISON")
print("="*80)
display(metrics_df)

# Save to CSV
metrics_df.to_csv(METRICS_DIR / "primary_metrics.csv", index=False)
print(f"\nSaved: {METRICS_DIR / 'primary_metrics.csv'}")

## 5. Multi-Agent Specific Metrics

In [ ]:
# Recovery Rate: queries that failed initially but succeeded after refinement
def compute_recovery_rate(states):
    recovered = 0
    total = 0
    
    for state in states:
        iterations = state.get('all_iterations', [])
        if len(iterations) > 1:
            total += 1
            # Check if first attempt failed but final succeeded
            first_failed = iterations[0].get('evaluation') in ['incorrect', 'error']
            final_succeeded = state.get('execution_success', False)
            
            if first_failed and final_succeeded:
                recovered += 1
    
    return recovered / total if total > 0 else 0

# First-attempt success rate
def compute_first_attempt_success(states):
    success = 0
    for state in states:
        iterations = state.get('all_iterations', [])
        if iterations and iterations[0].get('evaluation') == 'accept':
            success += 1
    return success / len(states) if states else 0

# Compute multi-agent specific metrics
recovery_rate = compute_recovery_rate(ma_states)
avg_iterations = multiagent_df['total_iterations'].mean()
first_attempt_success = compute_first_attempt_success(ma_states)

ma_metrics = pd.DataFrame({
    'Metric': [
        'Recovery Rate (%)',
        'Avg Iterations',
        'First-Attempt Success (%)',
        'Max Iterations Used',
        'Questions Requiring Refinement (%)'
    ],
    'Value': [
        f"{recovery_rate * 100:.1f}%",
        f"{avg_iterations:.2f}",
        f"{first_attempt_success * 100:.1f}%",
        multiagent_df['total_iterations'].max(),
        f"{(multiagent_df['total_iterations'] > 1).mean() * 100:.1f}%"
    ]
})

print("\n" + "="*80)
print("MULTI-AGENT SPECIFIC METRICS")
print("="*80)
display(ma_metrics)

ma_metrics.to_csv(METRICS_DIR / "multiagent_metrics.csv", index=False)
print(f"\nSaved: {METRICS_DIR / 'multiagent_metrics.csv'}")

## 6. Cost & Efficiency Metrics

In [ ]:
# Token usage comparison
baseline_total_tokens = baseline_df['total_tokens'].sum()
multiagent_total_tokens = multiagent_df['total_tokens'].sum()

baseline_avg_tokens = baseline_df['total_tokens'].mean()
multiagent_avg_tokens = multiagent_df['total_tokens'].mean()

# Latency comparison
baseline_total_time = baseline_df['elapsed_time'].sum()
multiagent_total_time = multiagent_df['elapsed_time'].sum()

baseline_avg_time = baseline_df['elapsed_time'].mean()
multiagent_avg_time = multiagent_df['elapsed_time'].mean()

cost_metrics = pd.DataFrame({
    'Metric': [
        'Total Tokens',
        'Avg Tokens per Question',
        'Token Overhead (%)',
        'Total Latency (s)',
        'Avg Latency per Question (s)',
        'Latency Overhead (%)'
    ],
    'Baseline': [
        f"{baseline_total_tokens:,}",
        f"{baseline_avg_tokens:,.0f}",
        "0%",
        f"{baseline_total_time:.1f}",
        f"{baseline_avg_time:.1f}",
        "0%"
    ],
    'Multi-Agent': [
        f"{multiagent_total_tokens:,}",
        f"{multiagent_avg_tokens:,.0f}",
        f"{((multiagent_total_tokens / baseline_total_tokens - 1) * 100):+.1f}%",
        f"{multiagent_total_time:.1f}",
        f"{multiagent_avg_time:.1f}",
        f"{((multiagent_total_time / baseline_total_time - 1) * 100):+.1f}%"
    ]
})

print("\n" + "="*80)
print("COST & EFFICIENCY METRICS")
print("="*80)
display(cost_metrics)

cost_metrics.to_csv(METRICS_DIR / "cost_efficiency.csv", index=False)
print(f"\nSaved: {METRICS_DIR / 'cost_efficiency.csv'}")

## 7. Stratified Analysis by Complexity

In [ ]:
# Merge baseline and multi-agent for stratified analysis
comparison_df = pd.DataFrame({
    'question_id': baseline_df['question_id'],
    'complexity': baseline_df['complexity'],
    'reasoning_level': baseline_df['reasoning_level'],
    'sublevel': baseline_df['sublevel'],
    'baseline_pass': baseline_df['pass_at_1'],
    'baseline_exec_success': baseline_df['execution_success'],
    'multiagent_pass': multiagent_df['pass_at_k'],
    'multiagent_exec_success': multiagent_df['execution_success'],
    'multiagent_iterations': multiagent_df['total_iterations']
})

# By Complexity
complexity_metrics = comparison_df.groupby('complexity').agg({
    'question_id': 'count',
    'baseline_pass': 'mean',
    'multiagent_pass': 'mean',
    'baseline_exec_success': 'mean',
    'multiagent_exec_success': 'mean',
    'multiagent_iterations': 'mean'
}).round(3)

complexity_metrics.columns = ['Count', 'Baseline Pass@1', 'MA Pass@k', 
                              'Baseline KG Valid', 'MA KG Valid', 'Avg Iterations']

# Calculate improvement
complexity_metrics['Pass@k Improvement'] = (
    (complexity_metrics['MA Pass@k'] - complexity_metrics['Baseline Pass@1']) * 100
).round(1)

print("\n" + "="*80)
print("STRATIFIED ANALYSIS: BY COMPLEXITY")
print("="*80)
display(complexity_metrics)

complexity_metrics.to_csv(METRICS_DIR / "stratified_by_complexity.csv")
print(f"\nSaved: {METRICS_DIR / 'stratified_by_complexity.csv'}")

## 8. Stratified Analysis by Reasoning Level

In [ ]:
# By Reasoning Level
reasoning_metrics = comparison_df.groupby('reasoning_level').agg({
    'question_id': 'count',
    'baseline_pass': 'mean',
    'multiagent_pass': 'mean',
    'baseline_exec_success': 'mean',
    'multiagent_exec_success': 'mean',
    'multiagent_iterations': 'mean'
}).round(3)

reasoning_metrics.columns = ['Count', 'Baseline Pass@1', 'MA Pass@k', 
                             'Baseline KG Valid', 'MA KG Valid', 'Avg Iterations']

reasoning_metrics['Pass@k Improvement'] = (
    (reasoning_metrics['MA Pass@k'] - reasoning_metrics['Baseline Pass@1']) * 100
).round(1)

print("\n" + "="*80)
print("STRATIFIED ANALYSIS: BY REASONING LEVEL")
print("="*80)
display(reasoning_metrics)

reasoning_metrics.to_csv(METRICS_DIR / "stratified_by_reasoning.csv")
print(f"\nSaved: {METRICS_DIR / 'stratified_by_reasoning.csv'}")

## 9. Stratified Analysis by Sublevel

In [ ]:
# By Sublevel
sublevel_metrics = comparison_df.groupby('sublevel').agg({
    'question_id': 'count',
    'baseline_pass': 'mean',
    'multiagent_pass': 'mean',
    'baseline_exec_success': 'mean',
    'multiagent_exec_success': 'mean',
    'multiagent_iterations': 'mean'
}).round(3)

sublevel_metrics.columns = ['Count', 'Baseline Pass@1', 'MA Pass@k', 
                            'Baseline KG Valid', 'MA KG Valid', 'Avg Iterations']

sublevel_metrics['Pass@k Improvement'] = (
    (sublevel_metrics['MA Pass@k'] - sublevel_metrics['Baseline Pass@1']) * 100
).round(1)

print("\n" + "="*80)
print("STRATIFIED ANALYSIS: BY SUBLEVEL")
print("="*80)
display(sublevel_metrics)

sublevel_metrics.to_csv(METRICS_DIR / "stratified_by_sublevel.csv")
print(f"\nSaved: {METRICS_DIR / 'stratified_by_sublevel.csv'}")

## 10. Visualization: Primary Metrics Comparison

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Pass@k comparison
metrics_to_plot = ['Pass@k', 'KG Valid@k', 'Exec Success']
baseline_vals = [
    baseline_df['pass_at_1'].mean() * 100,
    baseline_df['execution_success'].mean() * 100,
    baseline_df['execution_success'].mean() * 100
]
multiagent_vals = [
    multiagent_df['pass_at_k'].mean() * 100,
    multiagent_df['execution_success'].mean() * 100,
    multiagent_df['execution_success'].mean() * 100
]

x = np.arange(len(metrics_to_plot))
width = 0.35

ax[0].bar(x - width/2, baseline_vals, width, label='Baseline', alpha=0.8)
ax[0].bar(x + width/2, multiagent_vals, width, label='Multi-Agent', alpha=0.8)
ax[0].set_ylabel('Percentage (%)')
ax[0].set_title('Primary Metrics Comparison')
ax[0].set_xticks(x)
ax[0].set_xticklabels(metrics_to_plot)
ax[0].legend()
ax[0].grid(axis='y', alpha=0.3)

# Token usage comparison
systems = ['Baseline', 'Multi-Agent']
tokens = [baseline_avg_tokens, multiagent_avg_tokens]
ax[1].bar(systems, tokens, alpha=0.8, color=['#1f77b4', '#ff7f0e'])
ax[1].set_ylabel('Avg Tokens per Question')
ax[1].set_title('Token Efficiency Comparison')
ax[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(METRICS_DIR / 'primary_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {METRICS_DIR / 'primary_metrics_comparison.png'}")

## 11. Visualization: Stratified Performance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# By Complexity
complexity_plot = comparison_df.groupby('complexity')[['baseline_pass', 'multiagent_pass']].mean() * 100
complexity_plot.plot(kind='bar', ax=axes[0], alpha=0.8)
axes[0].set_title('Pass@k by Complexity')
axes[0].set_ylabel('Pass@k (%)')
axes[0].set_xlabel('Complexity Level')
axes[0].legend(['Baseline', 'Multi-Agent'])
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# By Reasoning Level
reasoning_plot = comparison_df.groupby('reasoning_level')[['baseline_pass', 'multiagent_pass']].mean() * 100
reasoning_plot.plot(kind='bar', ax=axes[1], alpha=0.8)
axes[1].set_title('Pass@k by Reasoning Level')
axes[1].set_ylabel('Pass@k (%)')
axes[1].set_xlabel('Reasoning Level')
axes[1].legend(['Baseline', 'Multi-Agent'])
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

# By Sublevel
sublevel_plot = comparison_df.groupby('sublevel')[['baseline_pass', 'multiagent_pass']].mean() * 100
sublevel_plot.plot(kind='bar', ax=axes[2], alpha=0.8)
axes[2].set_title('Pass@k by Sublevel')
axes[2].set_ylabel('Pass@k (%)')
axes[2].set_xlabel('Sublevel')
axes[2].legend(['Baseline', 'Multi-Agent'])
axes[2].grid(axis='y', alpha=0.3)
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig(METRICS_DIR / 'stratified_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {METRICS_DIR / 'stratified_performance.png'}")

## 12. Visualization: Iteration Distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Iteration count distribution
iteration_counts = multiagent_df['total_iterations'].value_counts().sort_index()
ax[0].bar(iteration_counts.index, iteration_counts.values, alpha=0.8, color='#2ca02c')
ax[0].set_xlabel('Number of Iterations')
ax[0].set_ylabel('Number of Questions')
ax[0].set_title('Distribution of Refinement Iterations')
ax[0].grid(axis='y', alpha=0.3)

# Success rate by iteration
success_by_iter = multiagent_df.groupby('total_iterations')['pass_at_k'].mean() * 100
ax[1].plot(success_by_iter.index, success_by_iter.values, marker='o', linewidth=2)
ax[1].set_xlabel('Number of Iterations')
ax[1].set_ylabel('Pass@k (%)')
ax[1].set_title('Success Rate by Iteration Count')
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(METRICS_DIR / 'iteration_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {METRICS_DIR / 'iteration_analysis.png'}")

## 13. Statistical Significance Testing

In [ ]:
from scipy import stats

# McNemar's test for paired binary outcomes
def mcnemar_test(baseline_success, multiagent_success):
    """
    McNemar's test for statistical significance of improvement.
    """
    # Contingency table
    both_success = sum((baseline_success) & (multiagent_success))
    baseline_only = sum((baseline_success) & (~multiagent_success))
    multiagent_only = sum((~baseline_success) & (multiagent_success))
    both_fail = sum((~baseline_success) & (~multiagent_success))
    
    # McNemar's test statistic
    if baseline_only + multiagent_only > 0:
        statistic = (abs(baseline_only - multiagent_only) - 1)**2 / (baseline_only + multiagent_only)
        p_value = 1 - stats.chi2.cdf(statistic, df=1)
    else:
        statistic = 0
        p_value = 1.0
    
    return {
        'statistic': statistic,
        'p_value': p_value,
        'both_success': both_success,
        'baseline_only': baseline_only,
        'multiagent_only': multiagent_only,
        'both_fail': both_fail
    }

# Test Pass@k improvement
pass_test = mcnemar_test(
    baseline_df['pass_at_1'].values.astype(bool),
    multiagent_df['pass_at_k'].values.astype(bool)
)

# Test Execution Success improvement
exec_test = mcnemar_test(
    baseline_df['execution_success'].values.astype(bool),
    multiagent_df['execution_success'].values.astype(bool)
)

print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE TESTING (McNemar's Test)")
print("="*80)

print("\nPass@k Improvement:")
print(f"  Statistic: {pass_test['statistic']:.4f}")
print(f"  P-value: {pass_test['p_value']:.4f}")
print(f"  Significant at α=0.05: {'Yes' if pass_test['p_value'] < 0.05 else 'No'}")
print(f"\n  Contingency:")
print(f"    Both succeeded: {pass_test['both_success']}")
print(f"    Baseline only: {pass_test['baseline_only']}")
print(f"    Multi-Agent only: {pass_test['multiagent_only']}")
print(f"    Both failed: {pass_test['both_fail']}")

print("\nExecution Success Improvement:")
print(f"  Statistic: {exec_test['statistic']:.4f}")
print(f"  P-value: {exec_test['p_value']:.4f}")
print(f"  Significant at α=0.05: {'Yes' if exec_test['p_value'] < 0.05 else 'No'}")

# Save statistical tests
stat_results = pd.DataFrame({
    'Test': ['Pass@k Improvement', 'Execution Success Improvement'],
    'Statistic': [pass_test['statistic'], exec_test['statistic']],
    'P-value': [pass_test['p_value'], exec_test['p_value']],
    'Significant (α=0.05)': [
        'Yes' if pass_test['p_value'] < 0.05 else 'No',
        'Yes' if exec_test['p_value'] < 0.05 else 'No'
    ]
})

stat_results.to_csv(METRICS_DIR / "statistical_tests.csv", index=False)
print(f"\nSaved: {METRICS_DIR / 'statistical_tests.csv'}")

## 14. Summary Report

In [ ]:
print("\n" + "="*80)
print("METRICS EVALUATION SUMMARY")
print("="*80)

print(f"\nDataset: {len(baseline_df)} questions")
print(f"Baseline: kg-axel (CoT + only_paths)")
print(f"Multi-Agent: 6-agent iterative refinement (max k=3)")

print("\n" + "-"*80)
print("PRIMARY RESULTS")
print("-"*80)
print(f"Pass@k:")
print(f"  Baseline: {baseline_df['pass_at_1'].mean()*100:.1f}%")
print(f"  Multi-Agent: {multiagent_df['pass_at_k'].mean()*100:.1f}%")
print(f"  Improvement: {(multiagent_df['pass_at_k'].mean() - baseline_df['pass_at_1'].mean())*100:+.1f}%")

print(f"\nKG Valid@k:")
print(f"  Baseline: {baseline_df['execution_success'].mean()*100:.1f}%")
print(f"  Multi-Agent: {multiagent_df['execution_success'].mean()*100:.1f}%")
print(f"  Improvement: {(multiagent_df['execution_success'].mean() - baseline_df['execution_success'].mean())*100:+.1f}%")

print("\n" + "-"*80)
print("MULTI-AGENT INSIGHTS")
print("-"*80)
print(f"Recovery Rate: {recovery_rate*100:.1f}%")
print(f"First-Attempt Success: {first_attempt_success*100:.1f}%")
print(f"Avg Iterations: {avg_iterations:.2f}")
print(f"Questions Requiring Refinement: {(multiagent_df['total_iterations'] > 1).sum()}/{len(multiagent_df)}")

print("\n" + "-"*80)
print("COST ANALYSIS")
print("-"*80)
print(f"Token Usage:")
print(f"  Baseline: {baseline_total_tokens:,} tokens ({baseline_avg_tokens:,.0f} avg/question)")
print(f"  Multi-Agent: {multiagent_total_tokens:,} tokens ({multiagent_avg_tokens:,.0f} avg/question)")
print(f"  Overhead: {((multiagent_total_tokens/baseline_total_tokens - 1)*100):+.1f}%")

print(f"\nLatency:")
print(f"  Baseline: {baseline_total_time:.1f}s ({baseline_avg_time:.1f}s avg/question)")
print(f"  Multi-Agent: {multiagent_total_time:.1f}s ({multiagent_avg_time:.1f}s avg/question)")
print(f"  Overhead: {((multiagent_total_time/baseline_total_time - 1)*100):+.1f}%")

print("\n" + "="*80)
print("OUTPUTS SAVED")
print("="*80)
print(f"  {METRICS_DIR / 'primary_metrics.csv'}")
print(f"  {METRICS_DIR / 'multiagent_metrics.csv'}")
print(f"  {METRICS_DIR / 'cost_efficiency.csv'}")
print(f"  {METRICS_DIR / 'stratified_by_complexity.csv'}")
print(f"  {METRICS_DIR / 'stratified_by_reasoning.csv'}")
print(f"  {METRICS_DIR / 'stratified_by_sublevel.csv'}")
print(f"  {METRICS_DIR / 'statistical_tests.csv'}")
print(f"  {METRICS_DIR / 'primary_metrics_comparison.png'}")
print(f"  {METRICS_DIR / 'stratified_performance.png'}")
print(f"  {METRICS_DIR / 'iteration_analysis.png'}")

print("\n" + "="*80)
print("Next: Run 04_efficiency_analysis.ipynb for detailed cost-benefit analysis")
print("="*80)